# Frequency bands and coverage by list family

This optional notebook looks at 2 things in the fixtures: how the rows fall
into frequency bands, and how much of the rows each list family covers. It
shows how to read -1 and null when you compute a coverage. It prints counts and
shares only and never a candidate.

In [ ]:
import os
import sys
from pathlib import Path

# kit/ holds the ceolkit package, and this notebook sits in kit/notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from ceolkit import figure, join_features, load, strata

# The fixtures are a sample of version 1.0's rows (kit/fixtures/). To use a
# full config, download its file and point SOURCE at it; with SOURCE = None,
# load() reads the hub, if the dataset is public or you are signed in to it.
SOURCE = Path(os.environ.get("CEOL_FIXTURES", "../fixtures"))

## Frequency bands

The `universe` config has no frequency column. The `features` config has
`wordfreq_zipf`, the wordfreq Zipf frequency, only for the strings wordfreq's
list holds. `join_features` adds it to the universe rows by `id`. A row with no
value (the string is not in wordfreq's list, or the row has more than 1 token
and so no features row) is in the band `no_value`, which is a band of its own.

In [ ]:
universe = load("universe", "v1.0", source=SOURCE)
features = load("features", "v1.0", source=SOURCE)
joined = join_features(universe, features, ["wordfreq_zipf"])

print(strata.frequency_band.definition)

In [ ]:
joined["frequency_band"] = strata.frequency_band.label_frame(joined)
bands = joined["frequency_band"].value_counts().reindex(strata.frequency_band.labels)
pd.DataFrame({"rows in the fixtures": bands, "share": (bands / bands.sum()).round(3)})

## Coverage by list family

For each single-token row and each of the 5 list families (WordNet, Wiktionary,
hunspell, ENABLE, SCOWL), `strata.family_cells` gives 1 (listed), 0 (not
listed), -1 (the family cannot hold the row's kind of string), or None (not
measured).

A coverage is the share of rows a family lists. Its denominator must be the
rows the family can hold: a -1 is not a "not listed", and counting it as one
makes a family look worse than it is on the strings it can hold. The last
column shows the difference.

In [ ]:
single = joined[joined["token_count"] == 1].copy()
cells = pd.DataFrame([strata.family_cells(r) for r in single.to_dict("records")], index=single.index)


def coverage(family):
    cell = cells[family]
    can_hold = cell.isin([0, 1])
    return {"listed": int((cell == 1).sum()),
            "cannot hold (-1)": int((cell == -1).sum()),
            "share of rows it can hold": round(float((cell == 1).sum() / can_hold.sum()), 3),
            "share if -1 is read as not listed": round(float((cell == 1).mean()), 3)}


table = pd.DataFrame({family: coverage(family) for family in cells.columns}).T
table.astype({"listed": int, "cannot hold (-1)": int})

The fixtures are an equal sample from each intake layer, so these shares
describe the fixtures. The release publishes its own counts by published set.
They are shares of all rows, including the multiword rows and the rows a set
cannot hold, so they are lower than a share of the rows a set can hold.

In [ ]:
published_sets = ["wordnet", "wiktionary_english", "hunspell_en_US", "hunspell_en_GB",
                  "hunspell_en_CA", "hunspell_en_AU", "enable1", "scowl"]
release_rows = figure("rows")
pd.DataFrame({
    "rows listed in the release": [figure(f"rows.listed_by.{name}") for name in published_sets],
    "share of all rows": [round(figure(f"rows.listed_by.{name}") / release_rows, 4) for name in published_sets],
}, index=published_sets)

## Frequency band and evidence tier together

Of the single-token rows in each frequency band, the share in each evidence
tier. A band with a value means wordfreq holds the string, and wordfreq is one
of the 2 corpora the tier reads, so rows with a value cannot fall in
`none_no_corpus`.

In [ ]:
single["evidence_tier"] = strata.evidence_tier.label_frame(single)
table = pd.crosstab(single["frequency_band"], single["evidence_tier"], normalize="index").round(3)
table.reindex([b for b in strata.frequency_band.labels if b in table.index])

In [ ]:
has_value = single["frequency_band"] != "no_value"
print("rows with a frequency value that are in none_no_corpus:",
      int((has_value & (single["evidence_tier"] == "none_no_corpus")).sum()))

## Next

To estimate these shares for the release and not for the fixtures, draw a
stratified sample (`02_draw_a_labeling_sample.ipynb`), weight it by its
manifest, and run the same code on it.